# 02 - Create EchoNet segmentation masks

This notebook first creates one binary LV mask for visual validation, then provides the full preprocessing cell that converts all traced frames into image-mask pairs.

In [ ]:
# Kaggle execution order: run after 01_explore_dataset.ipynb has verified paths.
# First run the single-frame sanity check. Only run full preprocessing after the overlay looks correct.
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from src.utils import (
    load_echonet_tables,
    preprocess_traced_frames,
    read_video_frame,
    save_mask_sanity_figure,
    tracing_group_to_mask,
    tracing_group_to_polygon,
    video_path_from_name,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

file_list, tracings = load_echonet_tables(RAW_DIR)

## Single-mask sanity check

Assumption: each traced frame group contains paired LV border points. The mask is generated by building a closed polygon from `(X1, Y1)` points followed by reversed `(X2, Y2)` points, then filling that polygon at the native frame resolution.

In [ ]:
selected_key = None
selected_rows = None

for key, rows in tracings.groupby(['FileName', 'Frame'], sort=True):
    if video_path_from_name(key[0], RAW_DIR).exists():
        selected_key = key
        selected_rows = rows
        break

assert selected_key is not None, 'No traced frame with an available video was found.'
file_name, frame_idx = selected_key
video_path = video_path_from_name(file_name, RAW_DIR)

frame = read_video_frame(video_path, int(frame_idx))
mask = tracing_group_to_mask(selected_rows, frame.shape[:2])
polygon = tracing_group_to_polygon(selected_rows)

print(f"Selected video: {file_name}")
print(f"Selected traced frame: {frame_idx}")
print(f"Frame shape: {frame.shape}")
print(f"Mask foreground pixels: {(mask > 0).sum():,}")

save_mask_sanity_figure(
    frame,
    mask,
    polygon,
    FIGURES_DIR / 'single_mask_sanity_check.png',
    title=f'{file_name} frame {frame_idx}',
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(frame, cmap='gray')
axes[0].plot(polygon[:, 0], polygon[:, 1], color='yellow', linewidth=1)
axes[0].set_title('Original frame + tracing')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Binary mask')
axes[2].imshow(frame, cmap='gray')
axes[2].imshow(mask, cmap='Reds', alpha=0.35)
axes[2].set_title('Overlay')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Full preprocessing and reusable dataset manifest

Run this section only after the single-mask overlay matches the LV. It writes grayscale frames to `data/processed/images/`, binary masks to `data/processed/masks/`, and a portable `metadata.csv` manifest. If a complete processed dataset already exists, regeneration is skipped.

In [ ]:
# Use MAX_SAMPLES = 16 only for a preprocessing smoke test.
# Keep MAX_SAMPLES = None to create the complete reusable Kaggle Dataset.
MAX_SAMPLES = None
FORCE_REGENERATE = False

IMAGE_DIR = PROCESSED_DIR / 'images'
MASK_DIR = PROCESSED_DIR / 'masks'
METADATA_PATH = PROCESSED_DIR / 'metadata.csv'
SUMMARY_PATH = PROCESSED_DIR / 'preprocess_summary.json'

existing_images = sorted(IMAGE_DIR.glob('*.png')) if IMAGE_DIR.exists() else []
existing_masks = sorted(MASK_DIR.glob('*.png')) if MASK_DIR.exists() else []
existing_image_stems = {path.stem for path in existing_images}
existing_mask_stems = {path.stem for path in existing_masks}
dataset_complete = bool(existing_images) and existing_image_stems == existing_mask_stems

if dataset_complete and not FORCE_REGENERATE:
    print('Complete processed image-mask dataset found; skipping regeneration.')
else:
    summary = preprocess_traced_frames(
        tracings=tracings,
        raw_dir=RAW_DIR,
        output_dir=PROCESSED_DIR,
        figures_dir=FIGURES_DIR,
        max_samples=MAX_SAMPLES,
        save_examples=8,
    )
    print(summary)

images = sorted(IMAGE_DIR.glob('*.png'))
masks = sorted(MASK_DIR.glob('*.png'))
image_by_stem = {path.stem: path for path in images}
mask_by_stem = {path.stem: path for path in masks}
paired_stems = sorted(image_by_stem.keys() & mask_by_stem.keys())

metadata_rows = []
for stem in paired_stems:
    video_id, frame_text = stem.rsplit('_frame', 1)
    metadata_rows.append({
        'video_id': video_id,
        'frame_idx': int(frame_text),
        'image_path': str(Path('images') / image_by_stem[stem].name),
        'mask_path': str(Path('masks') / mask_by_stem[stem].name),
    })

metadata = pd.DataFrame(
    metadata_rows,
    columns=['video_id', 'frame_idx', 'image_path', 'mask_path'],
)
metadata.to_csv(METADATA_PATH, index=False)

summary = {
    'image_count': len(images),
    'mask_count': len(masks),
    'paired_count': len(paired_stems),
    'counts_match': len(images) == len(masks) == len(paired_stems),
    'metadata_rows': len(metadata),
    'max_samples': MAX_SAMPLES,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

assert summary['counts_match'], 'Processed image and mask counts do not match.'
assert METADATA_PATH.exists(), 'metadata.csv was not created.'
assert SUMMARY_PATH.exists(), 'preprocess_summary.json was not created.'

print(f"Total processed images: {len(images):,}")
print(f"Total processed masks: {len(masks):,}")
print(f"Total image-mask pairs: {len(paired_stems):,}")
print(f"Metadata rows: {len(metadata):,}")
print(f"Processed dataset directory: {PROCESSED_DIR.resolve()}")
print('Files available for Kaggle Dataset export:')
for path in [IMAGE_DIR, MASK_DIR, METADATA_PATH, SUMMARY_PATH]:
    print(f'  - {path.resolve()}')

display(metadata.head())
summary

The complete `data/processed/` directory is now self-contained for conversion into a reusable Kaggle Dataset. Keep the notebook under `/kaggle/working` so images, masks, metadata, and the summary remain downloadable after completion.